# Session 14 · Clustering

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Data: supermarket customers (synthetic)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(14)
groups = [(2, 1500, 120), (8, 2500, 120), (4, 7000, 120)]     # (visits, basket Rs, count)
customers = pd.DataFrame(np.vstack([np.column_stack([rng.normal(v, 0.8, n), rng.normal(b, b * 0.12, n)])
                                    for v, b, n in groups]), columns=["visits_per_month", "avg_basket"])
print(customers.describe().round(1))

       visits_per_month  avg_basket
count             360.0       360.0
mean                4.7      3671.7
std                 2.7      2440.9
min                -0.3      1134.0
25%                 2.4      1636.9
50%                 3.9      2510.3
75%                 7.6      6409.1
max                 9.9      9228.6


## Example 1 - k-means

In [2]:
X = StandardScaler().fit_transform(customers[["visits_per_month", "avg_basket"]])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
customers["cluster"] = km.labels_
print(customers.groupby("cluster").mean().round(0))

         visits_per_month  avg_basket
cluster                              
0                     8.0      2519.0
1                     4.0      6982.0
2                     2.0      1513.0


## Example 2 - the elbow method

In [3]:
inertia = {k: round(KMeans(k, n_init=10, random_state=0).fit(X).inertia_, 1) for k in range(1, 9)}
print(inertia)
plt.plot(list(inertia), list(inertia.values()), marker="o"); plt.xlabel("k"); plt.ylabel("inertia"); plt.show()

{1: 720.0, 2: 372.1, 3: 47.7, 4: 37.7, 5: 31.4, 6: 24.9, 7: 20.8, 8: 18.2}


## Example 3 - silhouette score

In [4]:
sil = {k: round(silhouette_score(X, KMeans(k, n_init=10, random_state=0).fit_predict(X)), 3) for k in range(2, 7)}
print(sil)
assert max(sil, key=sil.get) == 3

{2: 0.547, 3: 0.806, 4: 0.664, 5: 0.535, 6: 0.43}


## Example 4 - why scaling matters

In [5]:
raw_labels = KMeans(3, n_init=10, random_state=0).fit_predict(customers[["visits_per_month", "avg_basket"]])
print("unscaled clusters, mean visits:", customers.groupby(raw_labels)["visits_per_month"].mean().round(1).tolist())
print("scaled clusters,   mean visits:", customers.groupby("cluster")["visits_per_month"].mean().round(1).tolist())

unscaled clusters, mean visits: [8.1, 3.9, 2.2]
scaled clusters,   mean visits: [8.1, 3.9, 2.0]


## Example 5 - hierarchical clustering

In [6]:
from scipy.cluster.hierarchy import dendrogram, linkage
Z = linkage(X[::9], method="ward")
dendrogram(Z, no_labels=True); plt.title("Dendrogram (every 9th customer)"); plt.show()
agg = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X)
print(pd.crosstab(customers["cluster"], agg))

col_0      0    1    2
cluster               
0          0  120    0
1        120    0    0
2          0    0  120


## Example 6 - DBSCAN

In [7]:
from sklearn.datasets import make_moons
Xm, _ = make_moons(300, noise=0.06, random_state=0)
Xm = np.vstack([Xm, [[2.5, 1.5], [-1.5, -0.8]]])         # two odd points
db = DBSCAN(eps=0.2, min_samples=5).fit(Xm)
print("labels found:", sorted(set(db.labels_)), "| noise points:", int((db.labels_ == -1).sum()))

labels found: [np.int64(-1), np.int64(0), np.int64(1)] | noise points: 2
